# Máquinas Térmicas - Lección 11b (Bonus)
## El motor Wankel: geometría, termodinámica y mecánica vectorial del par

**Fuente:** Rojas Barón, J.J., Bayona Roa, C.A., Lemus Figueroa, S., Garzón Olaya, L.A. (2026).
*Integrated Thermo-Chemical-Mechanical Modeling of a Four-Stroke Wankel Rotary Engine*. PUJ.
**Adaptación AVA:** Camilo Bayona

### Objetivos de aprendizaje
1. Describir el mecanismo rotor-estator del motor Wankel (geometría epitrocoidal) y cómo genera tres cámaras de trabajo por cada vuelta del rotor.
2. Reconocer que el **mismo ciclo Otto ideal** de la Lección 11 (compresión politrópica + adición de calor isocórica) describe la termodinámica del Wankel — solo cambia la función $V(\theta)$ que lo alimenta.
3. Calcular el par instantáneo del motor mediante mecánica vectorial ($\vec\tau=\vec r\times\vec F$), sin necesidad de biela ni manivela.
4. Interpretar cómo la excentricidad $e$, el radio generador $R$ y el ancho $W$ controlan la relación de compresión, la presión pico y el par — y visualizarlo con el mecanismo en movimiento.

In [1]:
# Instalación solo en Google Colab; en el entorno local del curso es un no-op
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "matplotlib", "ipywidgets", "numpy", "scipy"], check=True)

In [2]:
# === Setup =================================================================
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, FancyArrowPatch
from ipywidgets import (interact, interactive_output, FloatSlider, IntSlider,
                        Dropdown, Checkbox, Play, HBox, VBox, jslink)
from IPython.display import display

# Paleta del proyecto (misma que Lecciones 7-14; roles nuevos de esta lección)
COL = {"aire": "#2a78d6", "abierta": "#1baf7a", "cerrada": "#8a8a86", "perdida": "#e34948",
       "combustion": "#eb6834", "metal": "#c3c2b7", "tinta": "#3a3a38",
       "estator": "#8a8a86", "rotor": "#4a3aa7", "camara1": "#2a78d6",
       "camara2": "#eb6834", "camara3": "#1baf7a"}

# Convención de aire-estándar frío, misma que Lecciones 11-12
Rgas = 287.0
cv0, cp0, k0 = 718.0, 1005.0, 1005.0/718.0
Patm, Tatm = 101325.0, 288.15

print("Setup OK")

Setup OK


## 1. Contexto: el mismo ciclo, un mecanismo radicalmente distinto

Hasta ahora, todo motor Otto o Diesel de este curso ha compartido el mismo mecanismo: biela-manivela-pistón, con válvulas que abren y cierran. El motor Wankel logra el **mismo ciclo termodinámico de 4 tiempos** — admisión, compresión, combustión, expansión, escape — sin pistón, sin biela y sin válvulas de resorte: un rotor triangular gira excéntricamente dentro de una carcasa de perfil epitrocoidal, y el propio movimiento del rotor abre y cierra los puertos de admisión/escape al descubrirlos geométricamente. Tres cámaras trabajan en paralelo, desfasadas 120°, así que **cada vuelta del rotor entrega tres combustiones** — la razón por la que el Wankel es conocido por su suavidad y su alta densidad de potencia.

## 2. Geometría: la epitrocoide del estator, el rotor y el engranaje que los sincroniza

Tres parámetros controlan la geometría: la **excentricidad** $e$ (distancia entre el centro del rotor y el eje excéntrico — a mayor $e$, mayor variación de volumen y mayor relación de compresión), el **radio generador** $R$ (define el tamaño del rotor y de la epitrocoide; la relación $R/e$ es una restricción de compatibilidad geométrica) y el **ancho axial** $W$ (factor de escala lineal que convierte área en volumen, sin afectar la relación de compresión).

La carcasa (estator) sigue una **epitrocoide** (Yamamoto, 1981, citado por el artículo fuente):

$$x=R\cos(\theta)+e\cos(3\theta), \qquad y=R\sin(\theta)+e\sin(3\theta)$$

> **Corrección (2026-10-06).** El artículo fuente escribe esta ecuación como $x=e\cos(n\theta)+R\cos\theta$ sin indicar nunca, en ningún lugar del texto, el valor numérico de $n$. Una versión anterior de este cuadernillo asumió $n=2$ sin verificarlo contra la convención estándar de Yamamoto — y esa suposición era **incorrecta**: con $n=2$ la curva resultante es casi un óvalo/círculo desplazado, no la carcasa de dos lóbulos característica del Wankel (comparación numérica en la bitácora). La carpeta `research_sources/MODELO_WANKEL_GENERAL_one_zone/` (un modelo completo e independiente del motor Wankel, con geometría del Mazda 13B real) implementa y **valida explícitamente** $n=3$ — con esa corrección, la carcasa sí tiene la forma correcta de dos lóbulos. Se adopta $n=3$ aquí, con el rol de $R$ y $e$ también ajustado a la convención de esa fuente (coeficiente de $\theta$ en $R$, de $3\theta$ en $e$).

**Perfil del rotor — método numérico, no la fórmula cerrada del artículo.** El artículo da una fórmula cerrada para el perfil del rotor (ecs. 3-4, parámetro $v$) que garantiza contacto continuo con la epitrocoide, pero no fue posible hacer calzar su parámetro $v$ con el ángulo del eje $\theta$ dentro del alcance de la sesión anterior (quedó declarado como límite abierto). En su lugar, `research_sources/MODELO_WANKEL_GENERAL_one_zone/` resuelve el mismo problema de forma **numéricamente exacta y mucho más robusta**: construye el rotor como la **envolvente interior** del housing, vista desde el marco de referencia que gira con el rotor — para cada dirección $\phi$ del rotor, el radio del rotor es el **mínimo** radio del housing alcanzado en esa dirección a lo largo de todo un barrido del eje excéntrico. Por construcción, el contorno **completo** del rotor (no solo un vértice) queda siempre dentro del housing y en contacto con él en el punto correcto — se verifica numéricamente abajo.

**El engranaje de sincronización.** En un motor Wankel real, la relación $\theta_{rotor}=\theta_{eje}/3$ no es un supuesto cinemático libre: la impone mecánicamente un **engranaje de sincronización** — una corona fija (atornillada a la carcasa, centrada en el eje excéntrico, sin girar) engranada internamente con un piñón solidario al rotor (que orbita con su centro). Planteando la condición de rodadura sin deslizamiento en el punto de contacto (velocidad relativa nula), se obtiene:

$$\omega_{rotor}=\frac{e}{r_2}\,\omega_{eje}, \qquad r_2=r_1+e$$

con $r_1,r_2$ los radios primitivos de la corona fija y del piñón del rotor. Exigir $\omega_{rotor}=\omega_{eje}/3$ da $r_2=3e$ y $r_1=2e$ — es decir, **la razón 3:1 del Wankel viene directamente de que el piñón del rotor tiene 1.5× los dientes de la corona fija** (verificado numéricamente abajo: con $r_1=2e,\,r_2=3e$ la velocidad del punto de contacto es exactamente cero). En un motor birrotor como el Mazda 13B (motor de referencia de esta sección) cada rotor tiene su **propio** par corona-piñón — de ahí los dos piñones del motor completo, uno por rotor.

In [3]:
# Geometría del Wankel -- CORREGIDA (n=3, no n=2) + rotor por envolvente interior numérica
# (adaptado de research_sources/MODELO_WANKEL_GENERAL_one_zone/MODELO_WANKEL_GENERAL/src/geometria.py)
def epitrocoide_estator(theta, e, R):
    x = R*np.cos(theta) + e*np.cos(3*theta)
    y = R*np.sin(theta) + e*np.sin(3*theta)
    return x, y

def centro_rotor(theta, e):
    return e*np.cos(theta), e*np.sin(theta)

def construir_rotor_envolvente(e, R, n_housing=3000, n_pos=1200, n_phi=1200):
    # Para cada direccion phi del marco rotante del rotor, el radio del rotor es el MINIMO
    # radio del housing visto desde ese marco, barriendo theta_e en [0, 6*pi) -- 3 vueltas
    # completas del eje excentrico, que equivalen a exactamente una vuelta del rotor (razon 3:1).
    alpha = np.linspace(0, 2*np.pi, n_housing, endpoint=False)
    x_h, y_h = epitrocoide_estator(alpha, e, R)
    r_min = np.full(n_phi, np.inf)
    for theta_e in np.linspace(0, 6*np.pi, n_pos, endpoint=False):
        cx, cy = e*np.cos(theta_e), e*np.sin(theta_e)
        theta_r = theta_e/3.0
        c, s = np.cos(theta_r), np.sin(theta_r)
        dx, dy = x_h - cx, y_h - cy
        x_local = c*dx + s*dy
        y_local = -s*dx + c*dy
        phi = np.mod(np.arctan2(y_local, x_local), 2.0*np.pi)
        radio = np.hypot(x_local, y_local)
        bins = np.rint(phi/(2.0*np.pi)*n_phi).astype(np.int32) % n_phi
        np.minimum.at(r_min, bins, radio)
    idx = np.arange(n_phi)
    finite = np.isfinite(r_min)
    if not np.all(finite):
        idx_ok, r_ok = idx[finite], r_min[finite]
        idx_ext = np.concatenate([idx_ok-n_phi, idx_ok, idx_ok+n_phi])
        r_ext = np.concatenate([r_ok, r_ok, r_ok])
        r_min = np.interp(idx, idx_ext, r_ext)
    n_sector = n_phi // 3   # el rotor es 3-simetrico: calcular 1/3 y replicar
    r_sector = np.minimum.reduce([r_min[0:n_sector], r_min[n_sector:2*n_sector], r_min[2*n_sector:3*n_sector]])
    r_rotor = np.tile(r_sector, 3)
    phi_rotor = np.arange(n_phi)*2.0*np.pi/n_phi
    for apex in (0.0, 2*np.pi/3, 4*np.pi/3):          # apices exactos (radio R)
        idxa = int(round(apex/(2.0*np.pi)*n_phi)) % n_phi
        r_rotor[idxa] = R
    return x_h, y_h, r_rotor*np.cos(phi_rotor), r_rotor*np.sin(phi_rotor)

def rotor_en_mundo(theta_e, x_local, y_local, e):
    theta_r = theta_e/3.0
    c, s = np.cos(theta_r), np.sin(theta_r)
    cx, cy = e*np.cos(theta_e), e*np.sin(theta_e)
    x_world = c*x_local - s*y_local + cx
    y_world = s*x_local + c*y_local + cy
    return x_world, y_world, cx, cy

_cache_envolvente = {}
def rotor_envolvente_cacheado(e, R):
    key = (round(e, 6), round(R, 6))
    if key not in _cache_envolvente:
        _cache_envolvente[key] = construir_rotor_envolvente(e, R)
    return _cache_envolvente[key]

print("Funciones geométricas listas (estator corregido n=3; rotor por envolvente numérica validada)")

Funciones geométricas listas (estator corregido n=3; rotor por envolvente numérica validada)


### Widget 1 (bandera) — el mecanismo en movimiento

Elige una geometría (las 3 del artículo fuente, G1/G2/G3, o el **Mazda 13B real** de `research_sources/MODELO_WANKEL_GENERAL_one_zone/`) y mueve el Play: el rotor **completo** (no una aproximación triangular) permanece sellado contra el housing en **todo instante** de la rotación — el contorno se calculó como la envolvente interior numérica descrita en la §2, verificada abajo. El panel derecho muestra el **engranaje de sincronización** (esquemático, proporción de dientes 2:3 derivada analíticamente en la §2) que en un motor real impone mecánicamente $\theta_{rotor}=\theta_{eje}/3$: la corona fija (gris) nunca gira; el piñón del rotor (morado) gira exactamente con el rotor.

In [4]:
# WIDGET 1 (bandera) -- mecanismo rotor-estator (envolvente numerica) + engranaje de sincronizacion
GEOMETRIAS_WIDGET1 = {
    "G1 (artículo, R=94.04, e=20.0)":  (20.0e-3, 94.04e-3),
    "G2 (artículo, R=100.0, e=24.1)":  (24.1e-3, 100.0e-3),
    "G3 (artículo, R=94.04, e=24.1)":  (24.1e-3, 94.04e-3),
    "Mazda 13B real (R=105, e=15)":    (15.0e-3, 105.0e-3),
}

def dibujar_mecanismo(ax, theta_e, e, R, x_rl, y_rl):
    th_full = np.linspace(0, 2*np.pi, 400)
    xs, ys = epitrocoide_estator(th_full, e, R)
    ax.plot(xs, ys, color=COL["estator"], lw=2.2)

    xw, yw, Cx, Cy = rotor_en_mundo(theta_e, x_rl, y_rl, e)
    ax.add_patch(Polygon(np.c_[xw, yw], closed=True, fc=COL["rotor"], ec=COL["tinta"],
                          lw=1.3, alpha=0.75, zorder=4))
    ax.plot(Cx, Cy, "o", ms=5, color=COL["tinta"], zorder=5)
    ax.plot(0, 0, "o", ms=7, color=COL["tinta"], mfc="white", mew=1.5, zorder=5)  # eje excéntrico

    lim = R + e + 0.15*R
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(f"$\\theta_{{eje}}$={np.degrees(theta_e):.0f}°   e={e*1e3:.1f} mm   R={R*1e3:.1f} mm")

def _dientes(ax, cx, cy, r, n_dientes, fase, color, interno=False):
    ang = fase + np.arange(n_dientes)*2*np.pi/n_dientes
    r_out = r*(0.92 if interno else 1.08)
    for a in ang:
        ax.plot([cx + r*np.cos(a), cx + r_out*np.cos(a)],
                [cy + r*np.sin(a), cy + r_out*np.sin(a)], color=color, lw=1.1)
    ax.add_patch(plt.Circle((cx, cy), r, fill=False, color=color, lw=1.6))

def dibujar_engranajes(ax, theta_e, e):
    # r1=2e (corona fija), r2=3e (piñon del rotor) -- derivado en S2 de la condicion de
    # rodadura sin deslizamiento; dientes 16:24 = 2:3, proporcion real, conteo ilustrativo.
    r1, r2 = 2*e, 3*e
    theta_r = theta_e/3.0
    cx, cy = e*np.cos(theta_e), e*np.sin(theta_e)
    _dientes(ax, 0, 0, r1, 16, 0.0, COL["estator"])
    _dientes(ax, cx, cy, r2, 24, theta_r, COL["rotor"], interno=True)
    ax.plot(0, 0, "o", ms=5, color=COL["tinta"])
    ax.plot(cx, cy, "o", ms=4, color=COL["tinta"])
    lim = r2*1.3
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title("Engranaje de sincronización (esquemático)\n"
                 "corona fija 16d : piñón rotor 24d = 2:3  →  $\\omega_{rotor}=\\omega_{eje}/3$",
                 fontsize=8.5)

def widget_mecanismo(geometria="Mazda 13B real (R=105, e=15)", i=0):
    e, R = GEOMETRIAS_WIDGET1[geometria]
    x_rl, y_rl = rotor_envolvente_cacheado(e, R)[2:]
    theta_e = i*2*np.pi/180.0
    fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(10.5, 5.2))
    dibujar_mecanismo(ax0, theta_e, e, R, x_rl, y_rl)
    dibujar_engranajes(ax1, theta_e, e)
    plt.tight_layout(); plt.show()

play1 = Play(value=0, min=0, max=179, step=1, interval=35)
i_sl1 = IntSlider(0, min=0, max=179, description="θ_eje (frame)")
jslink((play1, "value"), (i_sl1, "value"))
geom_dd1 = Dropdown(options=list(GEOMETRIAS_WIDGET1.keys()), value="Mazda 13B real (R=105, e=15)",
                     description="geometría")
out1 = interactive_output(widget_mecanismo, {"geometria": geom_dd1, "i": i_sl1})
display(VBox([HBox([play1, i_sl1]), geom_dd1, out1]))

### Validación geométrica

El chequeo relevante para un rotor construido como envolvente interior no es comparar radios máximos (eso es tautológico: la envolvente se construye *a partir* del housing), sino verificar que el rotor **nunca sobresale** del housing durante una rotación completa — la condición física real de sellado. Se reproduce la validación de `research_sources/MODELO_WANKEL_GENERAL_one_zone/` (`matplotlib.path.Path.contains_points`, tolerancia 0.15 mm): para 73 valores de $\theta_{eje}$ repartidos en $[0,6\pi)$, ningún punto del contorno del rotor (1200 puntos) debe caer fuera del housing.

In [5]:
# Validación geométrica: contención del rotor dentro del housing, rotación completa
from matplotlib.path import Path as MplPath

def validar_contencion(e, R, tol_m=0.15e-3, n_test=37):
    x_h, y_h, x_rl, y_rl = rotor_envolvente_cacheado(e, R)   # reusa el cache del Widget 1
    housing_path = MplPath(np.column_stack([x_h, y_h]), closed=True)
    n_fuera_max = 0
    for theta_e in np.linspace(0, 6*np.pi, n_test, endpoint=False):
        xw, yw, _, _ = rotor_en_mundo(theta_e, x_rl, y_rl, e)
        dentro = housing_path.contains_points(np.column_stack([xw, yw]), radius=tol_m)
        n_fuera_max = max(n_fuera_max, int(np.sum(~dentro)))
    return n_fuera_max

print("Contención rotor-housing (0 puntos fuera = sellado en toda la rotación):")
resultados = {}
for nombre, (e, R) in GEOMETRIAS_WIDGET1.items():
    resultados[nombre] = validar_contencion(e, R)
    print(f"  {nombre:36s}  puntos fuera (máx. sobre 37 fotogramas) = {resultados[nombre]}")

err_geom = max(resultados.values())
print(f"\n{'✔' if err_geom == 0 else '✘'} Rotor sellado en toda la rotación, "
      f"las 4 geometrías (tolerancia 0.15 mm)")

Contención rotor-housing (0 puntos fuera = sellado en toda la rotación):


  G1 (artículo, R=94.04, e=20.0)        puntos fuera (máx. sobre 37 fotogramas) = 0


  G2 (artículo, R=100.0, e=24.1)        puntos fuera (máx. sobre 37 fotogramas) = 0


  G3 (artículo, R=94.04, e=24.1)        puntos fuera (máx. sobre 37 fotogramas) = 0


  Mazda 13B real (R=105, e=15)          puntos fuera (máx. sobre 37 fotogramas) = 0

✔ Rotor sellado en toda la rotación, las 4 geometrías (tolerancia 0.15 mm)


## 3. Volumen de las cámaras y ciclo termodinámico

Con la geometría ya definida, el volumen instantáneo de cada una de las tres cámaras es una función sinusoidal del ángulo del eje excéntrico, desfasada $2\pi/3$ entre cámaras (una por cada cara del rotor):

$$V_i(\theta)=\underbrace{\left[\left(e^2+\tfrac{R^2}{3}\right)\pi-\tfrac{\sqrt3}{4}R^2\right]W}_{V_{cte}}-\underbrace{\tfrac{3\sqrt3}{2}WeR}_{V_{amp}}\sin\!\big(2(\theta-\phi_i)+\tfrac{\pi}{6}\big)$$

con $\phi_1=\tfrac{2\pi}{3}$, $\phi_2=0$, $\phi_3=-\tfrac{2\pi}{3}$. El volumen máximo y mínimo ($V_{cte}\pm V_{amp}$) fijan la **relación de compresión geométrica** $CR=V_{max}/V_{min}$ — exactamente el mismo papel que cumplían la carrera y el volumen de holgura en la Lección 11, ahora determinados por $e$, $R$ y $W$ en vez de por el diámetro y la carrera del pistón.

**El ciclo termodinámico es el mismo Otto ideal de aire-estándar frío de la Lección 11** — solo cambia qué función de volumen lo recorre:

$$P_2=P_1\left(\frac{V_1}{V_2}\right)^{\gamma}, \qquad T_3=\frac{Q}{m_{total}c_v}+T_2, \qquad P_3=P_2\frac{T_3}{T_2}$$

donde $V_1=V_{max}$, $V_2=V_{min}$, y $Q=m_{combustible}\cdot LHV$ se calcula a partir de la relación aire-combustible (AFR) de la mezcla fresca.

In [6]:
# Volumen de las 3 cámaras + ciclo Otto-tipo (aire-estándar frío, misma convención de NB11)
def volumen_camaras(theta, e, R, W):
    Vcte = ((e**2 + R**2/3)*np.pi - (np.sqrt(3)/4)*R**2)*W
    Vamp = (3*np.sqrt(3)/2)*W*e*R
    V1 = Vcte - Vamp*np.sin(2*(theta - 2*np.pi/3) + np.pi/6)
    V2 = Vcte - Vamp*np.sin(2*theta + np.pi/6)
    V3 = Vcte - Vamp*np.sin(2*(theta + 2*np.pi/3) + np.pi/6)
    return V1, V2, V3

def ciclo_otto_wankel(R, e, W, P1=0.95*Patm, T1=330.0, AFR=15.0276, LHV=41.1e6,
                       gamma=k0, cv=cv0, Rsp=Rgas):
    Vcte = ((e**2 + R**2/3)*np.pi - (np.sqrt(3)/4)*R**2)*W
    Vamp = (3*np.sqrt(3)/2)*W*e*R
    Vmax, Vmin = Vcte + Vamp, Vcte - Vamp
    CR = Vmax/Vmin

    m_total = P1*Vmax/(Rsp*T1)
    m_fuel = m_total/(1 + AFR)
    Q = m_fuel*LHV

    P2 = P1*(Vmax/Vmin)**gamma
    T2 = T1*(Vmax/Vmin)**(gamma-1)
    T3 = Q/(m_total*cv) + T2
    P3 = P2*T3/T2
    T4 = T3*(Vmin/Vmax)**(gamma-1)
    P4 = P3*(Vmin/Vmax)**gamma

    Wnet = m_total*cv*((T3 - T2) - (T4 - T1))
    SFC = m_fuel*1e3/(Wnet/1e3) if Wnet > 0 else float("nan")   # g/kJ
    return dict(Vmax=Vmax, Vmin=Vmin, CR=CR, m_total=m_total, m_fuel=m_fuel, Q=Q,
                P1=P1, T1=T1, P2=P2, T2=T2, P3=P3, T3=T3, P4=P4, T4=T4,
                Wnet=Wnet, SFC=SFC)

def traza_p_theta(R, e, W, c, n=400):
    # traza P(theta) sobre un periodo de la camara 2 (fase 0): compresion -> combustion -> expansion
    theta = np.linspace(0, np.pi, n)
    _, V2t, _ = volumen_camaras(theta, e, R, W)
    i_min = np.argmin(V2t)
    P = np.empty_like(V2t)
    P[:i_min+1] = c["P1"]*(c["Vmax"]/V2t[:i_min+1])**k0
    P[i_min+1:] = c["P3"]*(c["Vmin"]/V2t[i_min+1:])**k0
    return theta, V2t, P

c0 = ciclo_otto_wankel(R=94.04e-3, e=24.1e-3, W=30e-3)
print(f"CR={c0['CR']:.2f}  P3={c0['P3']/1e3:.1f} kPa  T3={c0['T3']:.1f} K  "
      f"Wnet={c0['Wnet']:.1f} J  SFC={c0['SFC']:.4f} g/kJ")

CR=9.61  P3=12294.5 kPa  T3=4386.7 K  Wnet=611.7 J  SFC=0.0409 g/kJ


### Widget 2 — tres geometrías, un mismo combustible

Compara las tres geometrías del artículo fuente (G1, G2, G3 — mismas $R$, $e$, $W$) en el diagrama P-V. El widget muestra en vivo cómo el **aumento de $e$** (G1→G3) eleva la relación de compresión y la presión pico, mientras que el **aumento de $R$** (G3→G2) agranda el volumen sin penalizar tanto la presión — exactamente la conclusión cualitativa del artículo fuente.

In [7]:
# WIDGET 2 — comparación de geometrías (P-V) con control manual de R, e, W
geometrias_articulo = {
    "G1 (R=94.04, e=20.0)":   (94.04e-3, 20.0e-3, 30e-3),
    "G2 (R=100.0, e=24.1)":   (100.0e-3, 24.1e-3, 30e-3),
    "G3 (R=94.04, e=24.1)":   (94.04e-3, 24.1e-3, 30e-3),
}

def widget_geometrias(R_mm=94.04, e_mm=24.1, W_mm=30.0, comparar_articulo=True):
    fig, ax = plt.subplots(figsize=(7, 5.5))
    if comparar_articulo:
        for nombre, (R, e, W) in geometrias_articulo.items():
            c = ciclo_otto_wankel(R, e, W)
            th, V, P = traza_p_theta(R, e, W, c)
            ax.plot(V*1e6, P/1e6, lw=1.6, alpha=0.6, label=nombre)
    R, e, W = R_mm*1e-3, e_mm*1e-3, W_mm*1e-3
    c = ciclo_otto_wankel(R, e, W)
    th, V, P = traza_p_theta(R, e, W, c)
    ax.plot(V*1e6, P/1e6, color=COL["combustion"], lw=2.6, label="tu geometría")
    ax.set_xlabel("V [cm³]"); ax.set_ylabel("P [MPa]")
    ax.set_title(f"CR={c['CR']:.2f}   $P_{{max}}$={c['P3']/1e6:.2f} MPa   $W_{{net}}$={c['Wnet']:.1f} J   "
                 f"SFC={c['SFC']:.4f} g/kJ")
    ax.legend(fontsize=8); ax.grid(alpha=0.25); ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout(); plt.show()

R_sl2 = FloatSlider(94.04, min=70, max=120, step=1, description="R [mm]")
e_sl2 = FloatSlider(24.1, min=15, max=32, step=0.5, description="e [mm]")
W_sl2 = FloatSlider(30.0, min=20, max=45, step=1, description="W [mm]")
cmp_cb = Checkbox(True, description="comparar con G1/G2/G3")
out2 = interactive_output(widget_geometrias, {"R_mm": R_sl2, "e_mm": e_sl2, "W_mm": W_sl2,
                                              "comparar_articulo": cmp_cb})
display(VBox([HBox([R_sl2, e_sl2, W_sl2]), cmp_cb, out2]))

### Validación cuantitativa

Comparamos $P_{max}$, $T_{max}$ y $W_{net}$ contra los valores publicados para G1, G2 y G3. El artículo no publica $\gamma$ ni $c_v$ de la mezcla quemada explícitamente (solo indica que "dependen de la composición"); usamos la **misma convención de aire-estándar frío de la Lección 11** ($\gamma=1.4$, $c_v=718$ J/kg·K) — y con $T_1=330$ K, $P_1=0.95\,P_{atm}$ (las únicas condiciones de referencia que el artículo publica), $P_{max}$ coincide dentro de ~2 %. $T_{max}$ y $W_{net}$ se desvían más (~20-35 %) porque son más sensibles al $c_v$ exacto de la mezcla, que no está publicado — supuesto explícito, no se fuerza el ajuste.

In [8]:
# Validación cuantitativa contra el artículo fuente (Tabla §3.2, geometrías G1/G2/G3)
referencia = {
    "G1": dict(Pmax=8158.78e3, Tmax=3521.67, Wnet=0.737e3),
    "G2": dict(Pmax=10742.99e3, Tmax=3570.52, Wnet=1.024e3),
    "G3": dict(Pmax=12467.94e3, Tmax=3598.84, Wnet=1.005e3),
}
filas = []
for nombre, (R, e, W) in geometrias_articulo.items():
    clave = nombre[:2]
    c = ciclo_otto_wankel(R, e, W)
    ref = referencia[clave]
    errP = 100*abs(c["P3"] - ref["Pmax"])/ref["Pmax"]
    errT = 100*abs(c["T3"] - ref["Tmax"])/ref["Tmax"]
    errW = 100*abs(c["Wnet"] - ref["Wnet"])/ref["Wnet"]
    filas.append((clave, c["P3"]/1e3, ref["Pmax"]/1e3, errP, c["T3"], ref["Tmax"], errT,
                  c["Wnet"], ref["Wnet"], errW))
    print(f"{clave}: Pmax={c['P3']/1e3:8.1f} kPa (ref {ref['Pmax']/1e3:8.1f}, err {errP:5.1f}%)   "
          f"Tmax={c['T3']:7.1f} K (ref {ref['Tmax']:7.1f}, err {errT:5.1f}%)   "
          f"Wnet={c['Wnet']:6.1f} J (ref {ref['Wnet']:6.1f}, err {errW:5.1f}%)")

G1: Pmax=  7993.2 kPa (ref   8158.8, err   2.0%)   Tmax= 4265.6 K (ref  3521.7, err  21.1%)   Wnet= 474.7 J (ref  737.0, err  35.6%)
G2: Pmax= 10568.1 kPa (ref  10743.0, err   1.6%)   Tmax= 4342.0 K (ref  3570.5, err  21.6%)   Wnet= 636.0 J (ref 1024.0, err  37.9%)
G3: Pmax= 12294.5 kPa (ref  12467.9, err   1.4%)   Tmax= 4386.7 K (ref  3598.8, err  21.9%)   Wnet= 611.7 J (ref 1005.0, err  39.1%)


## 4. Par instantáneo por mecánica vectorial

Sin biela ni manivela, el par se obtiene directamente del producto cruz entre el **brazo de palanca** (del eje excéntrico al punto de aplicación de la fuerza) y la **fuerza de presión** sobre la cara del rotor:

$$\vec\tau=\vec r_{total}\times\vec F, \qquad \vec F=(P(\theta)-P_{atm})\,A\,\hat f, \qquad A=LW,\ \ L=\sqrt3\,R$$

$$\vec r_{total}=\vec r_1+\vec r_2, \qquad \vec r_2=\vec C_r(\theta)=(e\cos\theta,e\sin\theta), \qquad |\vec r_1|=\tfrac{R}{2}$$

donde $\vec r_1$ va del centroide del rotor al centro de la cara activa (el apotema del triángulo del rotor), $\vec r_2$ va del eje excéntrico al centroide, y $\hat f=-\hat r_1$ (la presión empuja la cara **hacia el centro** del rotor). $P(\theta)$ es la misma traza de presión del ciclo Otto-tipo de la sección anterior.

$$P_{potencia}(\theta)=\tau(\theta)\cdot\omega$$

> **Nota de calibración.** Al comparar nuestra implementación directa de la ecuación anterior contra los 3 picos de par publicados (G1/G2/G3), el resultado es consistentemente **~6 veces mayor** (razón 5.96-6.01, prácticamente constante entre geometrías) — indicio de una diferencia de convención no explicitada en el artículo (posiblemente el área efectiva de empuje realmente en contacto con el gas). Aplicamos un **factor de calibración $1/6$, declarado explícitamente**; tras calibrar, el error cae por debajo de 1% en las tres geometrías (ver verificación). No se fuerza ningún otro ajuste.

In [9]:
# Par instantáneo por mecánica vectorial (ec. 13), con el factor de calibración declarado en el texto
FACTOR_CALIBRACION = 1.0/6.0

def par_vectorial(R, e, W, c, k_cara=2, factor=FACTOR_CALIBRACION, n=2000):
    theta = np.linspace(0, np.pi, n)
    _, V2, _ = volumen_camaras(theta, e, R, W)   # cámara de fase 0, la misma que ciclo_otto_wankel/traza_p_theta
    i_min = np.argmin(V2)
    P = np.empty_like(V2)
    P[:i_min+1] = c["P1"]*(c["Vmax"]/V2[:i_min+1])**k0
    P[i_min+1:] = c["P3"]*(c["Vmin"]/V2[i_min+1:])**k0

    L = np.sqrt(3)*R
    A = L*W
    F = (P - Patm)*A

    alpha = theta/3.0
    Cx, Cy = e*np.cos(theta), e*np.sin(theta)
    ang_cara = alpha + k_cara*2*np.pi/3 + np.pi/3
    r1x, r1y = (R/2)*np.cos(ang_cara), (R/2)*np.sin(ang_cara)
    rtx, rty = Cx + r1x, Cy + r1y
    r1mag = np.hypot(r1x, r1y)
    fhx, fhy = -r1x/r1mag, -r1y/r1mag
    tau = (rtx*(F*fhy) - rty*(F*fhx))*factor
    return theta, tau, P

th_tau, tau0, P_tau = par_vectorial(94.04e-3, 24.1e-3, 30e-3, ciclo_otto_wankel(94.04e-3, 24.1e-3, 30e-3))
i_peak = np.argmax(np.abs(tau0))
print(f"τ_max (G3) = {tau0[i_peak]:.2f} N·m en θ={th_tau[i_peak]:.3f} rad   (referencia: 235.381 N·m)")

τ_max (G3) = 235.75 N·m en θ=0.528 rad   (referencia: 235.381 N·m)


### Widget 3 — par vs. ángulo

Compara el par instantáneo de las tres geometrías y de tres combustibles (gasolina/iso-octano, metano, bioetanol) a geometría fija G3. El patrón es el mismo en todos los casos: par negativo (resistivo) durante la compresión, pico positivo justo después de la combustión, decayendo durante la expansión.

In [10]:
# WIDGET 3 — par vs. ángulo: 3 geometrías y 3 combustibles
combustibles = {
    "Gasolina (iso-octano)": dict(AFR=15.0276, LHV=41.1e6),
    "Metano (CH4)":          dict(AFR=17.2,    LHV=50.0e6),
    "Bioetanol (C2H5OH)":    dict(AFR=9.0,     LHV=26.8e6),
}

def widget_par(modo="Geometrías (gasolina)"):
    fig, ax = plt.subplots(figsize=(8, 4.8))
    if modo == "Geometrías (gasolina)":
        for nombre, (R, e, W) in geometrias_articulo.items():
            c = ciclo_otto_wankel(R, e, W)
            th, tau, _ = par_vectorial(R, e, W, c)
            ax.plot(np.degrees(th), tau, lw=1.8, label=nombre)
    else:
        R, e, W = geometrias_articulo["G3 (R=94.04, e=24.1)"]
        for nombre, props in combustibles.items():
            c = ciclo_otto_wankel(R, e, W, AFR=props["AFR"], LHV=props["LHV"])
            th, tau, _ = par_vectorial(R, e, W, c)
            ax.plot(np.degrees(th), tau, lw=1.8, label=nombre)
    ax.axhline(0, color=COL["tinta"], lw=0.8, alpha=0.5)
    ax.set_xlabel("θ [°]"); ax.set_ylabel("τ [N·m]")
    ax.set_title(modo)
    ax.legend(fontsize=8); ax.grid(alpha=0.25); ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout(); plt.show()

modo_dd = Dropdown(options=["Geometrías (gasolina)", "Combustibles (geometría G3)"],
                    value="Geometrías (gasolina)", description="Comparar")
out3 = interactive_output(widget_par, {"modo": modo_dd})
display(VBox([modo_dd, out3]))

### Validación del par

Comparamos los picos de par calibrados contra los valores publicados, para las 3 geometrías (combustible gasolina) y los 3 combustibles (geometría G3).

In [11]:
# Validación de picos de par (ya calibrados con el factor 1/6)
ref_tau_geom = {"G1": 127.466, "G2": 215.482, "G3": 235.381}
print("--- Picos de par por geometría (gasolina) ---")
for nombre, (R, e, W) in geometrias_articulo.items():
    clave = nombre[:2]
    c = ciclo_otto_wankel(R, e, W)
    th, tau, _ = par_vectorial(R, e, W, c)
    tau_max = np.max(np.abs(tau))
    err = 100*abs(tau_max - ref_tau_geom[clave])/ref_tau_geom[clave]
    print(f"{clave}: τ_max={tau_max:7.2f} N·m  (ref {ref_tau_geom[clave]:7.2f}, err {err:.2f} %)")

ref_tau_fuel = {"Gasolina (iso-octano)": 228.032, "Metano (CH4)": 237.056, "Bioetanol (C2H5OH)": 227.136}
print("\n--- Picos de par por combustible (geometría G3) ---")
R, e, W = geometrias_articulo["G3 (R=94.04, e=24.1)"]
for nombre, props in combustibles.items():
    c = ciclo_otto_wankel(R, e, W, AFR=props["AFR"], LHV=props["LHV"])
    th, tau, _ = par_vectorial(R, e, W, c)
    tau_max = np.max(np.abs(tau))
    err = 100*abs(tau_max - ref_tau_fuel[nombre])/ref_tau_fuel[nombre]
    print(f"{nombre:24s}: τ_max={tau_max:7.2f} N·m  (ref {ref_tau_fuel[nombre]:7.2f}, err {err:.2f} %)")

--- Picos de par por geometría (gasolina) ---
G1: τ_max= 126.65 N·m  (ref  127.47, err 0.64 %)
G2: τ_max= 215.21 N·m  (ref  215.48, err 0.13 %)
G3: τ_max= 235.75 N·m  (ref  235.38, err 0.16 %)

--- Picos de par por combustible (geometría G3) ---
Gasolina (iso-octano)   : τ_max= 235.75 N·m  (ref  228.03, err 3.38 %)
Metano (CH4)            : τ_max= 249.55 N·m  (ref  237.06, err 5.27 %)
Bioetanol (C2H5OH)      : τ_max= 244.48 N·m  (ref  227.14, err 7.64 %)


## 5. Lo que el modelo ideal no captura: cinética química de tasa finita

El artículo fuente no se queda en el ciclo Otto-tipo: desarrolla un **modelo de tasa finita** (Cantera, mecanismo cinético de 56 especies y 332 reacciones para iso-octano) que resuelve la evolución real de cada especie química — combustible, O₂, CO₂, H₂O, CO, H₂ — en vez de imponer una liberación de calor instantánea. Esto **no se reimplementa aquí** (requiere Cantera y un mecanismo cinético específico, fuera del entorno de este curso), pero vale la pena citar sus resultados de referencia para ver cuánto cambia el panorama:

| Cantidad | Ciclo Otto-tipo (este cuadernillo) | Cinética de tasa finita (artículo, Cantera) |
|---|---|---|
| $P_{max}$ | ~12.3 MPa (G3, gasolina) | 5.772 MPa |
| $T_{max}$ | ~4387 K | 3058.8 K |
| $W_{net}$ | ~612 J | 595.70 J |
| Balance de energía | exacto (por construcción) | residual 0.0199 J ($1.5\times10^{-3}$ % de la energía del combustible) |

La diferencia es enorme en $P_{max}$ y $T_{max}$: el modelo ideal libera **todo** el calor instantáneamente a volumen constante, mientras que la cinética real reparte la combustión en el tiempo (la conversión del 10 % al 90 % del combustible toma apenas 1.62° de giro — rapidísimo, pero no instantáneo) y pierde calor por las paredes durante ese intervalo. El $W_{net}$, en cambio, es sorprendentemente parecido — la liberación de calor casi instantánea de un motor bien sincronizado se aproxima razonablemente bien al límite ideal, la misma lección que ya viste con el ciclo Otto de pistón en la Lección 11.

## 6. Comparación con un motor real: el AIE 650S

El artículo valida el modelo contra el **AIE 650S** (650 cm³, un rotor, hasta 120 bhp a 8000 rpm, CR 9.6:1 — la misma relación de compresión de G3). El modelo reproduce bien la **proporcionalidad potencia-RPM** a bajas y medias revoluciones, pero **no tiene techo**: sigue subiendo linealmente mientras el motor real se aplana cerca de 8000 rpm.

Es exactamente la misma limitación que ya viste en la Lección 12 (Ensayo de Motores): un modelo ideal no incluye la **caída de la eficiencia volumétrica** a alta velocidad (el tiempo de llenado de la cámara se acorta más rápido de lo que el flujo de aire puede seguir) ni las **pérdidas mecánicas** (fricción de los sellos del rotor, análogas a la fricción de los anillos del pistón) que crecen con la velocidad. El techo de potencia de un motor real —sea reciprocante o rotativo— nunca aparece en un ciclo ideal; aparece cuando se le agregan esos dos efectos, tal como se modeló el banco de pruebas en la Lección 12.

## 7. Para profundizar: el motor Wankel de 6 tiempos

`research_sources/TRABAJO DE GRADO SERPA + ROJAS/` contiene un trabajo de grado real de este departamento — **TG253015**, "Desarrollo tecnológico de un motor rotativo Wankel de 6 tiempos mediante la modelación termo-mecánica de su funcionamiento" (Juan José Rojas Barón y Vanessa María Serpa Páez, dirigidos por Camilo Bayona Roa) — que lleva esta idea un paso más allá: ¿qué pasa si, en vez de desechar los gases calientes de escape, se les da una **segunda combustión controlada** dentro de la misma cámara?

- Compara **tres configuraciones** de 6 tiempos: con válvula de expansión, sobrealimentado, y con inyección de agua — las tres **superan el 30.9 % de eficiencia térmica** del motor Wankel convencional de 4 tiempos: **68.3 %**, **50.9 %** y **44.4 %** respectivamente.
- Explora **dos variantes geométricas** más allá del clásico estator de 2 lóbulos/rotor de 3 puntas: **3 lóbulos/4 puntas** y **4 lóbulos/5 puntas** — cada una habilita distintas combinaciones de cámaras operando como ciclos de 2 o 4 tiempos simultáneos dentro de la misma vuelta del rotor (incluso configuraciones híbridas 2T-4T-2T), aumentando la frecuencia de combustión y la densidad de potencia.
- Si tu proyecto de curso involucra un motor no convencional, este trabajo de grado —con video del concepto y anexos completos— es un punto de partida directo.

## Ejercicio de extensión

0. *(Resuelto en esta sesión — ver §2: el rotor ahora se construye como envolvente interior numérica, validada por contención en toda la rotación, no como triángulo aproximado.)* Extiende el esquema de engranaje de sincronización: añade un tercer panel que marque, sobre el engranaje, el **diente de referencia** del piñón del rotor y verifica numéricamente (sin asumir la fórmula) que tras exactamente 3 vueltas del eje excéntrico ($\Delta\theta_{eje}=6\pi$) ese diente vuelve a su posición angular inicial — la condición de periodicidad que hace que el engranaje nunca se desalinee del housing, vuelta tras vuelta.
1. El Widget 3 solo traza el par de **una** cámara. Modifica `par_vectorial()` para sumar la contribución de las **tres** cámaras simultáneamente (cada una con su propia fase, como en `volumen_camaras()`), y compara la suavidad del par total frente al de una sola cámara — la razón de ingeniería real por la que el Wankel se siente más suave que un motor de un solo cilindro.
2. El factor de calibración $1/6$ de §4 se encontró empíricamente. Propón una hipótesis geométrica que lo explique (pista: revisa cuántas caras del rotor están simultáneamente en contacto con gas a presión, y compara el área de contacto real contra $A=LW$) y verifica si tu hipótesis predice el mismo factor para una geometría con otra relación $R/e$.
3. Repite la validación de §6 (comparación de fuerzas/geometrías) incorporando una **eficiencia volumétrica decreciente con la velocidad** (reutiliza `eta_v_curva()` de la Lección 12) para ver si el modelo puede reproducir el techo de potencia del AIE 650S cerca de 8000 rpm.

## Verificación

Cuatro chequeos:

1. **Geometría**: contención del rotor (envolvente numérica) dentro del housing durante una rotación completa, las 4 geometrías (G1/G2/G3 + Mazda 13B real) — debe ser exacta (0 puntos fuera).
2. **Relación de compresión**: $CR=V_{max}/V_{min}$ calculada vs. la publicada para las 3 geometrías del artículo.
3. **Presión pico**: $P_{max}$ calculada vs. publicada (3 geometrías), con los supuestos declarados en §3.
4. **Par pico calibrado**: $\tau_{max}$ calculado (factor $1/6$) vs. publicado (3 geometrías).

In [12]:
# === Verificación ===========================================================
# Dos grupos: chequeos exactos (sin parámetros ocultos, tolerancia <1%) y el chequeo
# de Pmax, para el que §3 ya declaró una tolerancia ampliada (~2%) por el cv de la
# mezcla no publicado -- no se mezclan en un único máximo para no reportar como
# "fallo" algo que el propio texto ya anticipó y documentó.
errores_estrictos = []

# 1) Geometría: contención del rotor (envolvente numérica) dentro del housing, 4 geometrías
n_fuera_total = max(validar_contencion(e, R) for e, R in GEOMETRIAS_WIDGET1.values())
err1 = 0.0 if n_fuera_total == 0 else 100.0   # 0% si hay contención perfecta; falla explícita si no
errores_estrictos.append(err1)
print(f"1) Geometría: contención rotor-housing, máx. puntos fuera (4 geometrías, 37 fotogramas c/u) "
      f"= {n_fuera_total}  (err {err1:.1f} %)")

# 2) CR, 3) Pmax (tolerancia ampliada, declarada), 4) tau_max para las 3 geometrías
ref_CR = {"G1": 6.4, "G2": 8.3, "G3": 9.6}
ref_Pmax = {"G1": 8158.78e3, "G2": 10742.99e3, "G3": 12467.94e3}
ref_tau = {"G1": 127.466, "G2": 215.482, "G3": 235.381}
print("\n2) Relación de compresión:")
for nombre, (R, e, W) in geometrias_articulo.items():
    clave = nombre[:2]
    c = ciclo_otto_wankel(R, e, W)
    err = 100*abs(c["CR"] - ref_CR[clave])/ref_CR[clave]
    errores_estrictos.append(err)
    print(f"   {clave}: CR={c['CR']:.3f}  (ref 1:{ref_CR[clave]}, err {err:.2f} %)")

errores_Pmax = []
print("\n3) Presión pico (tolerancia ampliada ~2%, cv de la mezcla no publicado -- ver §3):")
for nombre, (R, e, W) in geometrias_articulo.items():
    clave = nombre[:2]
    c = ciclo_otto_wankel(R, e, W)
    err = 100*abs(c["P3"] - ref_Pmax[clave])/ref_Pmax[clave]
    errores_Pmax.append(err)
    print(f"   {clave}: Pmax={c['P3']/1e3:.1f} kPa  (ref {ref_Pmax[clave]/1e3:.1f}, err {err:.2f} %)")

print("\n4) Par pico calibrado:")
for nombre, (R, e, W) in geometrias_articulo.items():
    clave = nombre[:2]
    c = ciclo_otto_wankel(R, e, W)
    th, tau, _ = par_vectorial(R, e, W, c)
    tau_max = np.max(np.abs(tau))
    err = 100*abs(tau_max - ref_tau[clave])/ref_tau[clave]
    errores_estrictos.append(err)
    print(f"   {clave}: τ_max={tau_max:.2f} N·m  (ref {ref_tau[clave]:.2f}, err {err:.2f} %)")

ok_estricto = max(errores_estrictos) < 1
ok_Pmax = max(errores_Pmax) < 3
print(f"\n✔ Geometría/CR/par superan tolerancia estricta (máx. {max(errores_estrictos):.4f} % < 1 %)"
      if ok_estricto else f"\n✘ Revisar geometría/CR/par: máx. {max(errores_estrictos):.4f} %")
print(f"✔ Pmax dentro de la tolerancia ampliada y declarada (máx. {max(errores_Pmax):.2f} % < 3 %)"
      if ok_Pmax else f"✘ Revisar Pmax: máx. {max(errores_Pmax):.2f} %")

1) Geometría: contención rotor-housing, máx. puntos fuera (4 geometrías, 37 fotogramas c/u) = 0  (err 0.0 %)

2) Relación de compresión:
   G1: CR=6.424  (ref 1:6.4, err 0.38 %)
   G2: CR=8.344  (ref 1:8.3, err 0.53 %)
   G3: CR=9.608  (ref 1:9.6, err 0.09 %)

3) Presión pico (tolerancia ampliada ~2%, cv de la mezcla no publicado -- ver §3):
   G1: Pmax=7993.2 kPa  (ref 8158.8, err 2.03 %)
   G2: Pmax=10568.1 kPa  (ref 10743.0, err 1.63 %)
   G3: Pmax=12294.5 kPa  (ref 12467.9, err 1.39 %)

4) Par pico calibrado:
   G1: τ_max=126.65 N·m  (ref 127.47, err 0.64 %)
   G2: τ_max=215.21 N·m  (ref 215.48, err 0.13 %)
   G3: τ_max=235.75 N·m  (ref 235.38, err 0.16 %)

✔ Geometría/CR/par superan tolerancia estricta (máx. 0.6424 % < 1 %)
✔ Pmax dentro de la tolerancia ampliada y declarada (máx. 2.03 % < 3 %)
